# Parallel Heston Calibration with ParFun + QuantLib

The **Heston model** is a widely-used stochastic volatility model in quantitative finance.
Calibrating it requires non-linear optimization against a grid of market-observed option prices.

A common technique is **multi-start optimization**: run the calibration from several different
initial guesses and pick the best result. Each run is independent, making this
embarrassingly parallel.

ParFun parallelises this with minimal code changes.

## Program Structure

![Program Structure](Heston_Calibration_Parfun_flowchart.svg)

# Imports and Environment

In [1]:
import time

import numpy as np
import QuantLib as ql

import parfun as pf

# Calibrate a single Heston model from one initial guess against a dense strike x maturity grid.

In [2]:
def calibrate_heston_once(initial_guess):
    """Calibrate a Heston model with a given initial guess.
    Uses a dense strike x maturity grid to create a realistic workload.
    Returns calibrated parameters and error."""
    spot = 100
    risk_free = 0.01
    dividend = 0.00

    rf_handle = ql.YieldTermStructureHandle(ql.FlatForward(0, ql.NullCalendar(), risk_free, ql.Actual365Fixed()))
    div_handle = ql.YieldTermStructureHandle(ql.FlatForward(0, ql.NullCalendar(), dividend, ql.Actual365Fixed()))

    process = ql.HestonProcess(
        rf_handle, div_handle,
        ql.QuoteHandle(ql.SimpleQuote(spot)),
        initial_guess[0], initial_guess[1], initial_guess[2],
        initial_guess[3], initial_guess[4],
    )

    model = ql.HestonModel(process)
    engine = ql.AnalyticHestonEngine(model)

    strikes = [70, 80, 85, 90, 95, 100, 105, 110, 115, 120, 130]
    maturities = [ql.Period(m, ql.Months) for m in [3, 6, 12, 18, 24]]
    base_vols = {
        70: 0.30, 80: 0.26, 85: 0.24, 90: 0.22, 95: 0.21,
        100: 0.20, 105: 0.21, 110: 0.22, 115: 0.24, 120: 0.26, 130: 0.30,
    }

    helpers = []
    for mat in maturities:
        for k in strikes:
            h = ql.HestonModelHelper(
                mat, ql.NullCalendar(), spot, k,
                ql.QuoteHandle(ql.SimpleQuote(base_vols[k])),
                rf_handle, div_handle,
            )
            h.setPricingEngine(engine)
            helpers.append(h)

    model.calibrate(helpers, ql.LevenbergMarquardt(), ql.EndCriteria(5000, 500, 1e-8, 1e-8, 1e-8))
    error = sum(h.calibrationError() for h in helpers)

    return list(model.params()), error

# Function to Parallelize

In [3]:
def calibrate_heston(initial_guesses):
    """Calibrate Heston model for each initial guess."""
    return [calibrate_heston_once(g) for g in initial_guesses]

# Parallelizing with ParFun

The only change needed is wrapping `calibrate_heston` with a `@pf.parfun` decorator.
The parallel version calls the **same function** — no logic is duplicated:

```diff
+ @pf.parfun(
+     split=pf.per_argument(initial_guesses=pf.py_list.by_chunk),
+     combine_with=pf.py_list.concat,
+     fixed_partition_size=1,
+ )
  def calibrate_heston_w_parfun(initial_guesses):
+     return calibrate_heston(initial_guesses)
  ...
+ with pf.set_parallel_backend_context("scaler_local", n_workers=4):
+     result = calibrate_heston_w_parfun(initial_guesses)
```

In [4]:
@pf.parfun(
    split=pf.per_argument(initial_guesses=pf.py_list.by_chunk),
    combine_with=pf.py_list.concat,
    fixed_partition_size=1,
)
def calibrate_heston_w_parfun(initial_guesses):
    return calibrate_heston(initial_guesses)

In [5]:
initial_guesses = [
    [0.1, 1.0, 0.05, 0.3, -0.5],
    [0.2, 0.5, 0.04, 0.4, -0.3],
    [0.05, 2.0, 0.07, 0.2, -0.7],
    [0.15, 1.5, 0.06, 0.25, -0.4],
    [0.08, 0.8, 0.03, 0.35, -0.6],
    [0.12, 1.2, 0.08, 0.15, -0.2],
    [0.18, 0.3, 0.05, 0.5, -0.8],
    [0.06, 1.8, 0.04, 0.28, -0.45],
]

start = time.time()
results_seq = calibrate_heston(initial_guesses)
seq_time = time.time() - start

start = time.time()
with pf.set_parallel_backend_context("scaler_local", n_workers=4):
    results_par = calibrate_heston_w_parfun(initial_guesses)
par_time = time.time() - start

print(f"Sequential: {seq_time:.2f}s")
print(f"Parallel:   {par_time:.2f}s")
print(f"Speedup:    {seq_time / par_time:.2f}x")

[INFO]2026-03-28 03:01:17+0800: logging to ('/dev/stdout',)
[INFO]2026-03-28 03:01:17+0800: ObjectStorageServer: start and listen to tcp://127.0.0.1:53129
[INFO]2026-03-28 03:01:17+0800: ObjectStorageServer: started
[INFO]2026-03-28 03:01:18+0800: logging to ('/dev/stdout',)
[INFO]2026-03-28 03:01:18+0800: use event loop: builtin
[INFO]2026-03-28 03:01:18+0800: ConfigController: scheduler_address = tcp://127.0.0.1:60193
[INFO]2026-03-28 03:01:18+0800: ConfigController: object_storage_address = tcp://127.0.0.1:53129
[INFO]2026-03-28 03:01:18+0800: ConfigController: monitor_address = None
[INFO]2026-03-28 03:01:18+0800: ConfigController: protected = True
[INFO]2026-03-28 03:01:18+0800: ConfigController: max_number_of_tasks_waiting = -1
[INFO]2026-03-28 03:01:18+0800: ConfigController: client_timeout_seconds = 60
[INFO]2026-03-28 03:01:18+0800: ConfigController: worker_timeout_seconds = 60
[INFO]2026-03-28 03:01:18+0800: ConfigController: object_retention_seconds = 60
[INFO]2026-03-28 03: